In [1]:
import pandas as pd

df = pd.read_csv("../../data/processed/noticias_panama_analizadas.csv")
df.info() # columnas disponibles para alimentar al modelo

<class 'pandas.DataFrame'>
RangeIndex: 1196 entries, 0 to 1195
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   medio               1196 non-null   str    
 1   titulo              1196 non-null   str    
 2   fecha               1196 non-null   str    
 3   categoria_original  1196 non-null   str    
 4   texto               1196 non-null   str    
 5   url                 1196 non-null   str    
 6   cantidad_palabras   1196 non-null   int64  
 7   categoria_predicha  1196 non-null   str    
 8   criticidad          1196 non-null   int64  
 9   sentimiento         1196 non-null   float64
 10  es_alerta           1196 non-null   bool   
 11  nivel_alerta        1196 non-null   int64  
 12  texto_lematizado    1196 non-null   str    
dtypes: bool(1), float64(1), int64(3), str(8)
memory usage: 5.8 MB


In [2]:
df["medio"].value_counts()

medio
Telemetro    650
TVN          347
La Prensa    199
Name: count, dtype: int64

In [3]:
# Estas columnas fueron descartadas pensando en implementar un modelo capaz de predecir el sentimiento en la noticia con información más allá de su propio contenido (metadatos)
# La idea es que se pueda responder a la pregunta: "Si tengo una noticia de X medio, de Y cantidad de palabaras, de Z categoría, etc..., ¿qué sentimiento es más probable que tenga?"
columnas_descartables = [
    "titulo",               # Descartado por ser contenido
    "texto",                # Descartado por ser contenido
    "url",                  # No es una variable relevante
    "categoria_predicha",   # Es un valor predicho por un LLM, no un metadato original de la noticia
    "es_alerta",            # Es un valor predicho por un LLM, no un metadato original de la noticia. Además, se calcula a partir del sentimiento, por lo que podría introducir sesgo
    "nivel_alerta",         # Es un valor predicho por un LLM, no un metadato original de la noticia. Además, se calcula a partir del sentimiento, por lo que podría introducir sesgo
    "texto_lematizado"      # Descartado por ser contenido
]
df = df.drop(columnas_descartables, axis=1)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1196 entries, 0 to 1195
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   medio               1196 non-null   str    
 1   fecha               1196 non-null   str    
 2   categoria_original  1196 non-null   str    
 3   cantidad_palabras   1196 non-null   int64  
 4   criticidad          1196 non-null   int64  
 5   sentimiento         1196 non-null   float64
dtypes: float64(1), int64(2), str(3)
memory usage: 85.9 KB


In [4]:
columnas_categoricas = ["medio", "categoria_original"]
df = pd.get_dummies(df, columns=columnas_categoricas, dtype=int)
df["fecha"] = pd.to_datetime(df["fecha"], format="%Y-%m-%d")
df["anio"] = df["fecha"].dt.year
df["mes"] = df["fecha"].dt.month
df = df.drop(["fecha"], axis=1)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1196 entries, 0 to 1195
Data columns (total 18 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   cantidad_palabras                       1196 non-null   int64  
 1   criticidad                              1196 non-null   int64  
 2   sentimiento                             1196 non-null   float64
 3   medio_La Prensa                         1196 non-null   int64  
 4   medio_TVN                               1196 non-null   int64  
 5   medio_Telemetro                         1196 non-null   int64  
 6   categoria_original_contenido-exclusivo  1196 non-null   int64  
 7   categoria_original_deportes             1196 non-null   int64  
 8   categoria_original_economia             1196 non-null   int64  
 9   categoria_original_entretenimiento      1196 non-null   int64  
 10  categoria_original_mundo                1196 non-null   int64  
 11  ca

In [5]:
df.to_csv("../../data/noticias_panama_preparadas.csv")